# AskMyBookmark — LangGraph Orchestrator Diagram

This notebook compiles the LangGraph orchestrator graph using minimal stub data
(no GitHub API call, no embedding) and renders the graph diagram.

Run all cells top-to-bottom. The final cell displays the compiled graph.

## 1. Setup

In [1]:
import sys
import os

# Add the repo root to sys.path so `app.orchestrator` is importable
REPO_ROOT = os.path.abspath("..")
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Load OPENAI_API_KEY (needed for ChatOpenAI object initialisation)
from dotenv import load_dotenv
load_dotenv(".env")

True

## 2. Build minimal stub data

The orchestrator graph requires a `search_df` (with `SearchArray`-indexed BM25 columns) and a
`vector_store`. We create one-row stub versions so the graph compiles without making any API
calls or loading real data.

In [2]:
import pandas as pd
from typing import List
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from searcharray import SearchArray
from app.orchestrator import _preprocess_text

# ── Minimal single-row DataFrame ──────────────────────────────────────────
stub_df = pd.DataFrame([{
    "id":                "stub-id",
    "repo":              "stub/repo",
    "description":       "stub description",
    "topics":            "stub",
    "topics_list":       ["stub"],
    "language":          "Python",
    "stars":             0,
    "url":               "https://github.com/stub/repo",
    "content":           "stub content",
    "curated_list_bm25": 0.0,
}])

# SearchArray BM25 columns required by MultiMatchBM25Retriever
stub_df["repo_idx"]        = SearchArray.index(stub_df["repo"],        tokenizer=_preprocess_text)
stub_df["description_idx"] = SearchArray.index(stub_df["description"], tokenizer=_preprocess_text)
stub_df["topics_idx"]      = SearchArray.index(stub_df["topics"],      tokenizer=_preprocess_text)
stub_df["content_idx"]     = SearchArray.index(stub_df["content"],     tokenizer=_preprocess_text)

# ── Stub vector store ──────────────────────────────────────────────────────
# EnsembleRetriever validates that its retrievers are real Runnable instances,
# so we need a proper BaseRetriever subclass rather than a MagicMock.
class _StubRetriever(BaseRetriever):
    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun
    ) -> List[Document]:
        return []

class _StubVectorStore:
    def as_retriever(self, **kwargs) -> _StubRetriever:
        return _StubRetriever()

stub_vector_store = _StubVectorStore()
print("Stub data ready.")

2026-03-19 15:35:51,008 - searcharray.indexing - INFO - Indexing begins w/ 4 workers
2026-03-19 15:35:51,009 - searcharray.indexing - INFO - 0 Batch Start tokenization
2026-03-19 15:35:51,009 - searcharray.indexing - INFO - Tokenizing 1 documents
2026-03-19 15:35:52,314 - searcharray.indexing - INFO - Tokenization -- vstacking
2026-03-19 15:35:52,315 - searcharray.indexing - INFO - Tokenization -- DONE
2026-03-19 15:35:52,315 - searcharray.indexing - INFO - Inverting docs->terms
2026-03-19 15:35:52,315 - searcharray.indexing - INFO - Encoding positions to bit array
2026-03-19 15:35:52,315 - searcharray.indexing - INFO - Batch tokenization complete
2026-03-19 15:35:52,316 - searcharray.indexing - INFO - (main thread) Processing 1 batch results
2026-03-19 15:35:52,316 - searcharray.indexing - INFO - Indexing from tokenization complete
2026-03-19 15:35:52,317 - searcharray.indexing - INFO - Indexing begins w/ 4 workers
2026-03-19 15:35:52,318 - searcharray.indexing - INFO - 0 Batch Start 

## 3. Compile the orchestrator graph

In [3]:
from app.orchestrator import build_orchestrator_graph

orchestrator = build_orchestrator_graph(
    search_df=stub_df,
    vector_store=stub_vector_store,
    # No checkpointer or query_cache_dir needed for diagram rendering
)

print("Graph compiled successfully.")
print("Nodes:", list(orchestrator.get_graph().nodes.keys()))

Graph compiled successfully.
Nodes: ['__start__', 'query_prep', 'refine_query', 'lexical_search', 'ensemble_search', 'merge_results', 'classify_curated', 'filter_results', 'rerank_results', 'generate_answer', 'human_feedback', '__end__']


## 4. Display the graph (horizontal layout, colour-coded by role)

In [4]:
import re
from IPython.display import HTML, display

# ── 1. Get the raw Mermaid string and flip to horizontal layout ────────────
mermaid_str = orchestrator.get_graph().draw_mermaid()
mermaid_lr  = re.sub(r'\bgraph TD\b', 'graph LR', mermaid_str)

# ── 2. Colour-code nodes by role ──────────────────────────────────────────
# Each group gets a classDef + a class assignment appended to the diagram.
#
#  Blue   — LLM nodes (call gpt-4o-mini)
#  Green  — Retrieval nodes (BM25 / dense search)
#  Purple — Processing nodes (merge, filter)
#  Amber  — human_feedback INTERRUPT (unique agentic step)

NODE_COLOURS = {
    # Default — all non-highlighted nodes get the same muted blue-grey
    "defaultNode": {
        "nodes": ["lexical_search", "ensemble_search", "merge_results",
                  "filter_results", "generate_answer", "human_feedback"],
        "style": "fill:#4a90d9,stroke:#2c5282,color:#ffffff,stroke-width:2px",
    },
    # ── Highlighted nodes (the 4 key steps) ─────────────────────────────────
    "queryPrepNode": {
        "nodes": ["query_prep"],
        "style": "fill:#0d9488,stroke:#0f766e,color:#ffffff,stroke-width:2px",
    },
    "refineNode": {
        "nodes": ["refine_query"],
        "style": "fill:#d97706,stroke:#b45309,color:#ffffff,stroke-width:2px",
    },
    "classifyNode": {
        "nodes": ["classify_curated"],
        "style": "fill:#7c3aed,stroke:#5b21b6,color:#ffffff,stroke-width:2px",
    },
    "rerankNode": {
        "nodes": ["rerank_results"],
        "style": "fill:#e11d48,stroke:#be123c,color:#ffffff,stroke-width:2px",
    },
}

colour_lines = []
for class_name, cfg in NODE_COLOURS.items():
    colour_lines.append(f"classDef {class_name} {cfg['style']}")
    colour_lines.append(f"class {','.join(cfg['nodes'])} {class_name}")

mermaid_coloured = mermaid_lr + "\n" + "\n".join(colour_lines)

# ── 3. Render in-browser (no external API call) ───────────────────────────
_graph_id = "ask-my-bookmark-graph"
_mermaid_escaped = mermaid_coloured.replace("`", r"\`")

display(HTML(f"""
<div style="background:#ffffff; padding:24px; border-radius:8px;
            overflow-x:auto; font-family:sans-serif;">
  <div id="{_graph_id}"></div>
  <p style="margin-top:16px; font-size:13px; color:#555;">
    <b style="color:#0d9488;">&#9632;</b> query_prep &nbsp;
    <b style="color:#d97706;">&#9632;</b> refine_query &nbsp;
    <b style="color:#7c3aed;">&#9632;</b> classify_curated &nbsp;
    <b style="color:#e11d48;">&#9632;</b> rerank_results &nbsp;
    <b style="color:#4a90d9;">&#9632;</b> generate_answer &nbsp;
    <b style="color:#48bb78;">&#9632;</b> Retrieval &nbsp;
    <b style="color:#64748b;">&#9632;</b> Processing &nbsp;
    <b style="color:#ed8936;">&#9632;</b> Human feedback (INTERRUPT)
  </p>
</div>
<script type="module">
  import mermaid from 'https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs';
  mermaid.initialize({{
    startOnLoad: false,
    theme: 'default',
    themeVariables: {{
      edgeLabelBackground: '#ffffff',
      lineColor: '#555555',
      fontSize: '15px'
    }}
  }});
  const {{ svg }} = await mermaid.render(
    '{_graph_id}-svg',
    `{_mermaid_escaped}`
  );
  document.getElementById('{_graph_id}').innerHTML = svg;
</script>
"""))

# --- Vertical PNG fallback (uses mermaid.ink API, may be slow) ---
# from IPython.display import Image
# display(Image(orchestrator.get_graph().draw_mermaid_png()))